# Thesis - Intelligent Reflecting Surface

## Dataset generation
I generate synthetic user positions in a 2D indoor area and, for each sample, compute an IRS phase-label vector by maximizing a simplified received-power metric with quantized phase candidates.

In [11]:
from itertools import product
import numpy as np

def dataset(samples=1000, size=8, K=4):
    """
    Generate synthetic UE positions and brute-force IRS phase labels.

    Args:
        samples: Number of dataset samples (UE positions).
        size: IRS side length. Total IRS elements are M = size**2.
        K: Number of quantized phase levels used in the search grid.

    Returns:
        positions: Array with shape (samples, 2) containing UE coordinates (x, y).
        theta_opt: Array with shape (samples, M) containing the best phase profile found
            for each sample according to the simplified received-power metric.
    """
    # IRS element positions on a 2D grid, with fixed height z = 1.5
    irs_pos = np.array(list(product(np.linspace(8.5, 9.5, size), np.linspace(3.5, 4.5, size), [1.5])))
    # Random UE positions in the area: x in [1, 8], y in [1, 7]
    positions = np.hstack((np.random.uniform(1, 8, (samples, 1)), np.random.uniform(1, 7, (samples, 1))))
    # IRS total number of reflecting elements
    M = size**2

    theta_opt = np.zeros((samples, M))
    for i in range(samples):
        p = np.array([positions[i, 0], positions[i, 1], 1.0])  # UE with z = 1

        # Simplified channels: pathloss approximation + random phase
        h_BI = 10 ** (-3.5 / 10) * np.exp(1j * np.random.uniform(0, 2 * np.pi, M))  # BS-IRS, 3.5 dB
        h_IU = np.exp(-np.linalg.norm(irs_pos - p, axis=1) / 10) * np.exp(1j * np.random.uniform(0, 2 * np.pi, M))  # IRS-UE

        best_pr, best_theta = 0, np.zeros(M)
        for theta_comb in product(np.linspace(0, 2*np.pi, K), M):
            theta = np.array(theta_comb)
            h_eff = np.dot(h_IU.conj() * h_BI, np.exp(1j * theta))
            pr = np.abs(h_eff)**2
            if pr > best_pr:
                best_pr, best_theta = pr, theta

        theta_opt[i] = best_theta

    return positions, theta_opt